In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

In [3]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [4]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


[]

### Methods

#### Get GDC programs - get_gdc_progams()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "facets": "program.name"  

#### Get valid primary sites - get_primary_sites()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "fields": "project_id,name,primary_site,disease_type"  

#### Get valid subtypes - get_subtypes()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> facets = "diagnoses.primary_diagnosis"  

#### For each subtype → get stages  - get_stages()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> "field": "diagnoses.ajcc_pathologic_stage" - AJCC diagnoses   

#### For each (subtype, stage) → get_samples()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> given pid, subtype, stage  
> from cases   
> "samples.sample_id", "samples.submitter_id", "samples.sample_type"  

#### get RNA-seq files - get_expression_files_given_samples()

> given: "field": "cases.case_id", "value": case_ids  
> end_point: url_gdc_files = "https://api.gdc.cancer.gov/files"  



### Get all programs

In [5]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File saved: /home/flavio/uv/perturb_agent/data/tcga/programs.txt


In [6]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [7]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [8]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(dfc))
dfc.head(12)


Table saved ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
12,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
7,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
17,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
0,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
1,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
23,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma
25,TCGA-LUSC,Bronchus and lung,TCGA-LUSC,"Squamous Cell Neoplasms, Adenomas and Adenocar...",Lung Squamous Cell Carcinoma
19,TCGA-MESO,"Bronchus and lung, Heart, mediastinum, and pleura",TCGA-MESO,Mesothelial Neoplasms,Mesothelioma
10,TCGA-CESC,"Cervix uteri, Ovary",TCGA-CESC,"Squamous Cell Neoplasms, Cystic, Mucinous and ...",Cervical Squamous Cell Carcinoma and Endocervi...


In [9]:
dfc.tail(3)

,pid,primary_site,project_id,disease_type,name
29,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
3,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
31,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

In [10]:
i = 5
pid = dfc.iloc[i].pid
primary_site = dfc.iloc[i].primary_site

print(i, pid, primary_site)

5 TCGA-BRCA Breast


- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [11]:
force=False
verbose=True

i = 5
pid = dfc.iloc[i].pid
primary_site = dfc.iloc[i].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))
df_subt

5 TCGA-BRCA Breast
.......

👉 Returned 1098 / Total paginated 1098
Table saved ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'
Table saved ((12, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/subtype_for_TCGA-BRCA.tsv'
df_cases 1098 df_subt 12


,pid,subtype_global,tumor_class,subtype_tissue,sstage,n
0,TCGA-BRCA,other,other,other,II,447
1,TCGA-BRCA,lobular,other,breast_lobular,missing,223
2,TCGA-BRCA,other,other,other,III,154
3,TCGA-BRCA,other,other,other,I,141
4,TCGA-BRCA,other,other,other,missing,85
5,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,missing,19
6,TCGA-BRCA,other,other,other,IV,16
7,TCGA-BRCA,ductal,adenocarcinoma,breast_ductal,missing,6
8,TCGA-BRCA,ductal,other,breast_ductal,II,2
9,TCGA-BRCA,clear_cell,other,clear_cell,II,2


In [12]:
cols = ["pid", "case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,pid,case_id,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-BRCA,e3935ce4-64d3-4a66-ba11-d308b844b410,other,other,other,unknown
1,TCGA-BRCA,e3b555aa-7f0a-49c6-9b13-182c61a144c1,other,other,other,Stage IIIA
2,TCGA-BRCA,17ca61a2-607a-45ff-88fa-ef72e80bf891,other,other,other,Stage IA
3,TCGA-BRCA,7d681cc6-689d-41c8-9e84-e13733089ec9,other,other,other,Stage IIB
4,TCGA-BRCA,17f275c1-a0d4-487d-8f02-ea279584b4cd,other,other,other,Stage IA
5,TCGA-BRCA,189e1f27-7738-413a-a4d4-97d41d592a13,other,other,other,Stage IIA
6,TCGA-BRCA,95c53ecf-d8f1-4bcb-9b1a-c9a0542939f0,other,other,other,Stage IA
7,TCGA-BRCA,c0f88fc5-56b0-4255-bbd1-9a21ced8b37b,other,other,other,Stage IIIA
8,TCGA-BRCA,0f64edec-0f1f-4025-8a53-75f9534f7828,other,other,other,Stage IIA
9,TCGA-BRCA,c25898b4-eb33-42f4-bf3f-bf532a929e7d,lobular,other,breast_lobular,unknown


### Get all cases and their classifications

In [ ]:

force=False
verbose=True

run_all = False

if run_all:

    for i in range(len(dfc)):
        print(i, end=' ')
        pid = dfc.iloc[i].pid
        primary_site = dfc.iloc[i].primary_site

        print(pid, primary_site)

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)



0 TCGA-ACC Adrenal gland
..

👉 Returned 92 / Total paginated 92
Table saved ((92, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-ACC.tsv'
Table saved ((4, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/subtype_for_TCGA-ACC.tsv'
1 TCGA-PCPG Adrenal gland, Retroperitoneum and peritoneum, Other endocrine glands and related structures, Other and ill-defined sites, Connective, subcutaneous and other soft tissues, Spinal cord, cranial nerves, and other parts of central nervous system, Heart, mediastinum, and pleura
..

👉 Returned 179 / Total paginated 179
Table saved ((179, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-PCPG.tsv'
Table saved ((5, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/subtype_for_TCGA-PCPG.tsv'
2 TCGA-BLCA Bladder
....

👉 Returned 412 / Total paginated 412
Table saved ((412, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BLCA.tsv'
Table saved ((13, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/subtype_for_TCGA-BL

### Search for samples and files for each case given

- input: pid, subtype_global, tumor_class, subtype_tissue, and stage

In [ ]:
i = 6
df_subt.iloc[i]

In [ ]:
force=False
verbose=True

i = 6
row = df_subt.iloc[i]

pid = row.pid
subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue
sstage = row.sstage

s_case = f"{pid} subtype '{subtype_global}' tumor '{tumor_class}' subtype_tissue '{subtype_tissue}' stage '{sstage}'"
print(s_case)

df_sample = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                             tumor_class=tumor_class, subtype_tissue=subtype_tissue,
                                             sstage=sstage, batch_size=200,
                                             force=force, verbose=verbose)

print(len(df_sample))

df_sample.head(3)


In [ ]:
np.unique(df_sample.data_type)

### Testing loop

In [ ]:
N_cases = 54
batch_cases=5

ini = -batch_cases
end = 0

while(True):
    ini += batch_cases
    end += batch_cases
                
    if ini >= N_cases: break
    if end > N_cases:
        end = N_cases

    print(f"{ini}-{end}")


### Get Any table (for tumor and control)

In [ ]:
force=False
verbose=True

i=0
row = df_sample.iloc[i]

pid = row.pid
case_id = row.case_id
file_type = row.data_type
file_id = row.file_id
sample_type = row.sample_type
stage = row.stage


print(f"{pid}, {case_id}, {file_type}, {file_id}")

dft = gdc.get_table_given_fileID(pid=pid, case_id=case_id, 
                                 file_type=file_type, sample_type=sample_type, stage=stage, file_id=file_id, 
                                 force=force, verbose=verbose)
print(len(dft))
dft.head(6)

### Get Expression table (for tumor and control)

#### 🧪 Mental model (important)

| Step	|  Needs strand? | Output | Column |
|---------|---------|---------|---------|
| Read assignment	|  YES |  counts | stranded_first |
| Normalization	| NO | TPM / FPKM | both: unstranded |
| Biological meaning | NO | gene expression| both: unstranded |

In [ ]:
file_type = 'Gene Expression Quantification'

force=True
verbose=True

dft = pd.DataFrame()

df2 = df_sample[df_sample.data_type == file_type]

for i, row in df2.iterrows():
    pid = row.pid
    case_id = row.case_id
    file_type = row.data_type
    file_id = row.file_id
    sample_type = row.sample_type
    stage = row.stage

    print(f"{i}) {pid}, {case_id}, {file_type}, {file_id}")

    dft = gdc.get_table_given_fileID(pid=pid, case_id=case_id, 
                                     sample_type=sample_type, stage=stage, 
                                     file_type=file_type, file_id=file_id, 
                                     force=force, verbose=verbose)


dft.head(6)

